# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

My chosen lane is **Refresh / Content Opportunity Scoring**.

I frame this lane mainly as a **ranking / scoring** task. The product question is not "Can I train a model?" The product question is: **which content pages should a reviewer inspect first when review time is limited?**

The output should be a page-level opportunity score, sorted from highest priority to lowest priority. A reviewer can then inspect the top pages and decide whether to refresh, expand, rewrite metadata, protect, prune, or monitor each page.

There is also a classification piece in the starter data, because the current proxy target can be written as a yes/no decline flag. But the decision we care about is ranked action priority, so I will judge the work using ranking metrics at the top of the queue.


In [1]:
from pathlib import Path

import pandas as pd

DATA_PATH = Path("../../data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(DATA_PATH)

print(f"Starter rows: {len(df):,}")
print(f"Starter columns: {df.shape[1]:,}")
print(f"Pseudonymized clients: {df['client_id'].nunique():,}")
print(f"Unique content IDs: {df['content_id'].nunique():,}")


Starter rows: 30,000
Starter columns: 44
Pseudonymized clients: 32
Unique content IDs: 30,000


## 2. Target or proxy

For the starter notebook, my provisional proxy target is:

```text
is_declining_proxy = trend_direction == "down"
```

This label means the page's most recent 30-day impressions were more than 20% lower than the previous 30-day window, using the starter dataset's existing `trend_direction` field.

This is useful for learning the workflow, but it is **not the ideal capstone target**. It is a defined proxy from the current snapshot, not a future observed outcome. Because of that, `trend_direction` and `trend_pct` must not be used as normal model features when predicting this proxy. In the later warehouse work, a stronger target would be something like:

```text
features from a prior window -> decline, recovery, or opportunity in a future window
```

For this Week 2 framing, I will use the starter proxy only to sketch what the target column looks like and to make the ML loop concrete.


In [2]:
df_task = df.copy()
df_task["is_declining_proxy"] = df_task["trend_direction"].eq("down").astype(int)

target_summary = pd.Series(
    {
        "declining_proxy_pages": int(df_task["is_declining_proxy"].sum()),
        "non_declining_proxy_pages": int((1 - df_task["is_declining_proxy"]).sum()),
        "declining_proxy_rate_pct": round(100 * df_task["is_declining_proxy"].mean(), 1),
    }
)

target_summary.to_frame("value")


,value
declining_proxy_pages,16262.0
non_declining_proxy_pages,13738.0
declining_proxy_rate_pct,54.2


## 3. Success metric

The main success metric should be **precision@K**, especially **precision@50**.

This fits the real decision: a reviewer usually cannot review every page, so the most important question is whether the top of the ranked queue contains pages that are actually worth inspection. Precision@50 asks: out of the top 50 recommended pages, how many match the target or proxy?

I would also track:

- **average precision**, because the full order of the ranked list matters.
- **recall at K**, if missing important decline/opportunity pages becomes the larger business risk.
- **manual top-20 review**, because reason codes and actionability matter even when a metric looks good.

For this lane, generic accuracy is not enough. If most value comes from the first 20 or 50 pages, the metric should evaluate the first 20 or 50 pages.


In [3]:
# A tiny metric sketch using a transparent starter score, not a final model.
# This shows how precision@K connects to a ranked review queue.

scored = df_task.assign(
    visible=df_task["impressions_90d"].ge(500).astype(int),
    low_ctr_visible=(
        df_task["impressions_90d"].ge(500)
        & df_task["avg_position"].gt(0)
        & df_task["avg_position"].le(20)
        & df_task["ctr"].lt(0.5)
    ).astype(int),
    stale_visible=(
        df_task["days_since_last_update"].ge(180)
        & df_task["impressions_90d"].ge(500)
    ).astype(int),
)

scored["simple_review_score"] = (
    0.50 * scored["visible"]
    + 0.35 * scored["low_ctr_visible"]
    + 0.15 * scored["stale_visible"]
)

top_50 = scored.sort_values("simple_review_score", ascending=False).head(50)
precision_at_50 = top_50["is_declining_proxy"].mean()

pd.Series(
    {
        "baseline_precision_at_50_against_proxy": round(float(precision_at_50), 3),
        "top_50_declining_proxy_pages": int(top_50["is_declining_proxy"].sum()),
        "top_50_total_pages": len(top_50),
    }
).to_frame("value")


,value
baseline_precision_at_50_against_proxy,0.76
top_50_declining_proxy_pages,38.00
top_50_total_pages,50.00


## 4. The unit of analysis, as a real dataframe

The unit of analysis is **one content page**.

Each row in the dataframe below is one pseudonymized content item. The IDs are safe for grouping and validation, but they should not be used as model features. The useful feature candidates are observable signals that would help a reviewer prioritize work: impressions, clicks, CTR, position, sessions, engagement, content age, freshness, and content type.

The last column, `is_declining_proxy`, sketches the starter target. It is included here so the ML loop is visible, but it should be handled carefully because it comes from `trend_direction`.


In [4]:
unit_columns = [
    "content_id",
    "client_id",
    "content_type",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "sessions_90d",
    "engagement_rate",
    "content_age_days",
    "days_since_last_update",
    "trend_direction",
    "is_declining_proxy",
]

lane_slice = df_task.loc[:, unit_columns].head(10)
lane_slice


,content_id,client_id,content_type,impressions_90d,clicks_90d,ctr,avg_position,sessions_90d,engagement_rate,content_age_days,days_since_last_update,trend_direction,is_declining_proxy
0,content_304f48230142,client_f369cb89fc,keyword article,3803,29,0.76,10.6,17,5.88,187,20,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,7,0.05,20.3,9,0.00,445,25,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,11,0.09,36.5,11,0.00,141,20,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,58,0.49,6.2,78,1.28,463,22,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,24,0.13,44.0,145,0.00,263,14,down,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,3970,1,0.03,8.5,5,0.00,147,20,down,1
6,content_9a34b442b552,client_8722616204,keyword article,20,0,0.00,7.0,1,0.00,90,20,down,1
7,content_a63219c6e95a,client_19581e27de,keyword article,1724,1,0.06,21.2,28,3.57,445,22,stable,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,32574,29,0.09,46.0,68,5.88,90,20,down,1
9,content_c27558df2b0c,client_19581e27de,keyword article,1240,2,0.16,4.9,3,0.00,257,104,down,1


## 5. Why ML beats a fixed rule here

A fixed rule is a necessary baseline, but it is too rigid to be the whole solution. A rule like "review pages with at least 500 impressions and CTR below 0.5%" can find obvious candidates, but it cannot easily balance all the signals that matter at the same time.

For this lane, the priority of a page depends on combinations of signals:

- A page with strong impressions and low CTR may need title, snippet, or intent review.
- A page with good position and declining movement may need protection or refresh.
- A page with weak engagement may need content structure review even if clicks look acceptable.
- A stale page with demand may be more urgent than a stale page nobody sees.
- A low-volume page may look bad only because the numbers are noisy.

ML can help if it learns which combinations of observable signals are associated with useful review priority and produces a better top-K queue than a transparent baseline. The model still does not replace human judgment. The output should include reason codes and careful confidence labels so a reviewer can understand why a page was surfaced.


In [5]:
feature_roles = pd.DataFrame(
    [
        {"column": "impressions_90d", "role": "feature candidate", "note": "search demand / visibility"},
        {"column": "clicks_90d", "role": "feature candidate", "note": "observed search traffic"},
        {"column": "ctr", "role": "feature candidate", "note": "x100 percentage; compare with position context"},
        {"column": "avg_position", "role": "feature candidate with gotcha", "note": "0 means no position data"},
        {"column": "sessions_90d", "role": "feature candidate", "note": "engagement context"},
        {"column": "engagement_rate", "role": "feature candidate", "note": "x100 percentage"},
        {"column": "content_age_days", "role": "feature candidate", "note": "freshness / lifecycle context"},
        {"column": "days_since_last_update", "role": "feature candidate", "note": "staleness context"},
        {"column": "trend_direction", "role": "proxy source only", "note": "do not use as a feature for this proxy"},
        {"column": "trend_pct", "role": "leakage risk", "note": "defines trend direction; exclude from proxy model features"},
        {"column": "content_id", "role": "grouping only", "note": "pseudonymized ID, not a feature"},
        {"column": "client_id", "role": "validation grouping only", "note": "use for client holdout, not as a feature"},
    ]
)

feature_roles


,column,role,note
0,impressions_90d,feature candidate,search demand / visibility
1,clicks_90d,feature candidate,observed search traffic
2,ctr,feature candidate,x100 percentage; compare with position context
3,avg_position,feature candidate with gotcha,0 means no position data
4,sessions_90d,feature candidate,engagement context
5,engagement_rate,feature candidate,x100 percentage
6,content_age_days,feature candidate,freshness / lifecycle context
7,days_since_last_update,feature candidate,staleness context
8,trend_direction,proxy source only,do not use as a feature for this proxy
9,trend_pct,leakage risk,defines trend direction; exclude from proxy mo...


## Self-check

Before submitting, I checked each line honestly:

- [✓] I named the ML task type: ranking / scoring, with a starter classification proxy.
- [✓] I named the target/proxy: `is_declining_proxy = trend_direction == "down"`.
- [✓] I explained that the starter proxy is not the ideal future-window capstone target.
- [✓] I named a success metric: precision@50, with average precision and manual top-20 review as supporting checks.
- [✓] I showed the unit of analysis as a real dataframe: one row equals one pseudonymized content page.
- [✓] I tied the output to a real content action: review for refresh, metadata, expansion, protection, pruning, or monitoring.
- [✓] I explained why ML can improve over a fixed rule while still requiring human review.
- [✓] I noted leakage risks around `trend_direction`, `trend_pct`, and pseudonymized IDs.
